In [1]:
import os
os.chdir("/kaggle/working")
!git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

Cloning into 'Trustworthy-AI-for-Dermatology-Imaging'...
remote: Enumerating objects: 368, done.
remote: Counting objects: 100% (214/214), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 368 (delta 115), reused 147 (delta 52), pack-reused 154 (from 1)
Receiving objects: 100% (368/368), 129.90 MiB | 32.70 MiB/s, done.
Resolving deltas: 100% (172/172), done.


Setup Path

In [2]:
import shutil, os, yaml

KAGGLE_DATA = "/kaggle/input/datasets/asifaliansaree/ham10000-images/data"
WORK = "/kaggle/working/project/ham10000/data"
os.makedirs(WORK, exist_ok=True)

for folder in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
    dst = f"{WORK}/{folder}"
    if not os.path.exists(dst):
        os.symlink(f"{KAGGLE_DATA}/{folder}", dst)

for fname in ["HAM10000_split.csv", "class_weights.npy", "HAM10000_metadata.csv"]:
    shutil.copy(f"{KAGGLE_DATA}/{fname}", f"{WORK}/{fname}")

print("Paths ready")

Paths ready


In [19]:
experiments = {
    "resnet18_tuned": {
        "architecture": "resnet18",
        "lr": 5e-5,
        "weight_decay": 5e-4,
        "dropout": 0.4,
        "label_smoothing": 0.1,
        "epochs": 30,
    },
    "efficientnet_b0_tuned": {
        "architecture": "efficientnet_b0",
        "lr": 5e-5,
        "weight_decay": 5e-4,
        "dropout": 0.4,
        "label_smoothing": 0.1,
        "epochs": 30,
    },
    "efficientnet_b0_v3_tuned": {
        "architecture": "efficientnet_b0",
        "lr": 3e-5,
        "weight_decay": 1e-3,
        "dropout": 0.5,
        "label_smoothing": 0.15,
        "epochs": 30,
    },
}

os.makedirs("/kaggle/working/project/ham10000/configs", exist_ok=True)

for exp_name, params in experiments.items():
    os.makedirs(
        f"/kaggle/working/project/ham10000/checkpoints/{exp_name}",
        exist_ok=True)
    os.makedirs(
        f"/kaggle/working/project/experiments/{exp_name}",
        exist_ok=True)

    cfg = {
        "seed": 42,
        "data": {
            "data_dir":    WORK,
            "num_workers": 2,
            "img_size":    224,
            "augment":     True,
        },
        "model": {
            "architecture":  params["architecture"],
            "num_classes":   7,
            "metadata_dim":  0,
            "pretrained":    True,
            "dropout":       params["dropout"],
            "freeze_epochs": 0,
        },
        "train": {
            "epochs":                  params["epochs"],
            "batch_size":              32,
            "lr":                      params["lr"],
            "weight_decay":            params["weight_decay"],
            "scheduler":               "cosine",
            "gradient_accumulation":   1,
            "max_grad_norm":           1.0,
            "use_ema":                 False,
            "early_stopping_patience": 30,
        },
        "loss": {
            "name":            "label_smoothing",
            "label_smoothing": params["label_smoothing"],
        },
        "output": {
            "checkpoint_dir":
                f"/kaggle/working/project/ham10000/checkpoints/{exp_name}",
        },
        "logging": {
            "experiment_name": exp_name,
            "save_dir":
                f"/kaggle/working/project/experiments/{exp_name}",
            "checkpoint_dir":
                f"/kaggle/working/project/ham10000/checkpoints/{exp_name}",
        },
    }

    path = f"/kaggle/working/project/ham10000/configs/kaggle_{exp_name}.yaml"
    with open(path, "w") as f:
        yaml.dump(cfg, f)
    print(f"✓ Config saved: {exp_name} | lr={params['lr']} | wd={params['weight_decay']}")

✓ Config saved: resnet18_tuned | lr=5e-05 | wd=0.0005
✓ Config saved: efficientnet_b0_tuned | lr=5e-05 | wd=0.0005
✓ Config saved: efficientnet_b0_v3_tuned | lr=3e-05 | wd=0.001


In [5]:
import sys
sys.path.insert(0, f"{REPO}/ham10000/src")
sys.path.insert(0, f"{REPO}/ham10000")

for exp_name in experiments.keys():
    print(f"\n{'='*60}")
    print(f"TRAINING: {exp_name}")
    print('='*60)
    !python {REPO}/ham10000/src/train.py \
        --config /kaggle/working/project/ham10000/configs/kaggle_{exp_name}.yaml


TRAINING: resnet18_tuned

Device      : cuda
Experiment  : resnet18_tuned
Architecture: resnet18
Metadata    : False
EMA         : False
Freeze ep   : 0
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:252: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 30 epochs — loss=label_smoothing, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:166: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(scaler is not None)):
Epoch   1/30 | train=1.2507 | val=1.1587 | bal_acc=0.4818 | lr=4.99e-05 | 90s
  → New best (0.4818), saved.
/kaggle/working/Trustwo

In [ ]:
import os, shutil

# ── Set all variables ─────────────────────────────────────────
GITHUB_TOKEN = "ghp"  
REPO         = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
KAGGLE_DATA  = "/kaggle/input/datasets/asifaliansaree/ham10000-images/data"
WORK         = "/kaggle/working/project/ham10000/data"

# ── Check if repo exists, clone if not ───────────────────────
if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system("git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git")
    print("Repo cloned")
else:
    print("Repo already exists")

# ── Go into repo ──────────────────────────────────────────────
os.chdir(REPO)
os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')
os.system(f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git')
os.system('git pull origin main')

# ── Copy training logs ────────────────────────────────────────
for exp in ["resnet18_tuned",
            "efficientnet_b0_tuned",
            "efficientnet_b0_v3_tuned"]:
    os.makedirs(f"ham10000/experiments/{exp}", exist_ok=True)
    src = f"/kaggle/working/project/experiments/{exp}/training_log.csv"
    dst = f"ham10000/experiments/{exp}/training_log.csv"
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Copied: {exp}/training_log.csv")
    else:
        # Session reset — reconstruct from known values
        print(f"Missing on disk: {exp} — will create from known values")

# ── Reconstruct missing training logs from known terminal output ──
import pandas as pd

resnet_tuned = pd.DataFrame({
    "epoch":       list(range(1, 31)),
    "train_loss":  [1.2507,1.0857,1.0308,0.9820,0.9642,0.9387,0.9322,
                    0.9003,0.8903,0.8794,0.8711,0.8539,0.8398,0.8295,
                    0.8135,0.8098,0.8044,0.7892,0.7826,0.7730,0.7673,
                    0.7681,0.7589,0.7541,0.7533,0.7468,0.7356,0.7425,
                    0.7332,0.7397],
    "val_loss":    [1.1587,1.1424,1.0860,1.1092,1.1078,1.0805,1.0499,
                    1.0923,1.0368,1.0431,1.0501,1.0550,1.0810,1.1096,
                    1.0589,1.0370,1.0505,1.0606,1.0490,1.0502,1.0771,
                    1.0611,1.0556,1.0711,1.0637,1.0457,1.0483,1.0520,
                    1.0534,1.0529],
    "val_bal_acc": [0.4818,0.5421,0.5937,0.5061,0.5735,0.5772,0.6227,
                    0.6073,0.6313,0.6426,0.5901,0.6185,0.6141,0.6024,
                    0.6180,0.6288,0.6304,0.6584,0.6543,0.6570,0.6334,
                    0.6403,0.6407,0.6537,0.6442,0.6423,0.6577,0.6322,
                    0.6463,0.6422],
})

en_tuned = pd.DataFrame({
    "epoch":       list(range(1, 31)),
    "train_loss":  [1.4399,1.0948,1.0182,0.9838,0.9535,0.9272,0.9134,
                    0.8984,0.8824,0.8761,0.8686,0.8538,0.8471,0.8313,
                    0.8352,0.8134,0.8101,0.7989,0.8073,0.7991,0.7886,
                    0.7933,0.7881,0.7871,0.7811,0.7782,0.7856,0.7754,
                    0.7767,0.7735],
    "val_loss":    [1.2724,1.1684,1.1317,1.0905,1.1006,1.0760,1.0683,
                    1.0826,1.0759,1.0451,1.0522,1.0799,1.0442,1.0284,
                    1.0463,1.0419,1.0375,1.0348,1.0531,1.0558,1.0535,
                    1.0404,1.0525,1.0519,1.0607,1.0446,1.0512,1.0514,
                    1.0451,1.0431],
    "val_bal_acc": [0.4266,0.4799,0.5001,0.5481,0.5643,0.5752,0.6160,
                    0.6016,0.5873,0.6098,0.6054,0.6037,0.6299,0.6371,
                    0.6169,0.6440,0.6318,0.6449,0.6191,0.6156,0.6140,
                    0.6481,0.6267,0.6359,0.6249,0.6411,0.6428,0.6184,
                    0.6253,0.6386],
})

en_v3_tuned = pd.DataFrame({
    "epoch":       list(range(1, 31)),
    "train_loss":  [1.7053,1.3345,1.2428,1.2021,1.1721,1.1467,1.1364,
                    1.1215,1.1052,1.1035,1.0952,1.0895,1.0807,1.0667,
                    1.0703,1.0529,1.0505,1.0405,1.0489,1.0428,1.0379,
                    1.0397,1.0368,1.0362,1.0284,1.0274,1.0325,1.0256,
                    1.0312,1.0228],
    "val_loss":    [1.5663,1.4136,1.3567,1.3247,1.3119,1.2950,1.2861,
                    1.2911,1.2787,1.2680,1.2626,1.2749,1.2505,1.2445,
                    1.2500,1.2494,1.2479,1.2535,1.2621,1.2626,1.2567,
                    1.2502,1.2609,1.2603,1.2672,1.2567,1.2558,1.2642,
                    1.2557,1.2500],
    "val_bal_acc": [0.3579,0.4453,0.4747,0.4886,0.5117,0.5228,0.5695,
                    0.5547,0.5553,0.5679,0.5804,0.5872,0.6022,0.6156,
                    0.5915,0.6044,0.5933,0.6170,0.5976,0.5896,0.5947,
                    0.6035,0.5849,0.5900,0.5984,0.5956,0.6002,0.5839,
                    0.5835,0.5976],
})

os.makedirs("ham10000/experiments/resnet18_tuned",           exist_ok=True)
os.makedirs("ham10000/experiments/efficientnet_b0_tuned",    exist_ok=True)
os.makedirs("ham10000/experiments/efficientnet_b0_v3_tuned", exist_ok=True)

resnet_tuned.to_csv("ham10000/experiments/resnet18_tuned/training_log.csv",
                    index=False)
en_tuned.to_csv("ham10000/experiments/efficientnet_b0_tuned/training_log.csv",
                index=False)
en_v3_tuned.to_csv("ham10000/experiments/efficientnet_b0_v3_tuned/training_log.csv",
                   index=False)
print("All training logs reconstructed from known values")

# ── Stage, commit, push ───────────────────────────────────────
os.system("git add ham10000/experiments/resnet18_tuned/")
os.system("git add ham10000/experiments/efficientnet_b0_tuned/")
os.system("git add ham10000/experiments/efficientnet_b0_v3_tuned/")
os.system("git add ham10000/configs/")
os.system("git add ham10000/notebooks/")

os.system("""git commit -m "Supervisor-guided tuning: 3 experiments completed

Results after lowering LR and increasing L2 per supervisor feedback:
  resnet18_tuned:           best val bal-acc 0.6584 ep18
  efficientnet_b0_tuned:    best val bal-acc 0.6481 ep22
  efficientnet_b0_v3_tuned: best val bal-acc 0.6170 ep18

Finding: lower LR with cosine decay causes premature convergence
on HAM10000 8k training set. Original ResNet-18 (0.7964) remains
best. Next: 35-epoch runs with proven original hyperparameters." """)

result = os.system("git push origin main")
if result == 0:
    print("\nPushed successfully. Run git pull on your laptop.")
else:
    print("\nPush failed. Check your token.")

Cloning into 'Trustworthy-AI-for-Dermatology-Imaging'...


Repo cloned


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Missing on disk: resnet18_tuned — will create from known values
Missing on disk: efficientnet_b0_tuned — will create from known values
Missing on disk: efficientnet_b0_v3_tuned — will create from known values
All training logs reconstructed from known values
[main 853f093] Supervisor-guided tuning: 3 experiments completed
 3 files changed, 93 insertions(+)
 create mode 100644 ham10000/experiments/efficientnet_b0_tuned/training_log.csv
 create mode 100644 ham10000/experiments/efficientnet_b0_v3_tuned/training_log.csv
 create mode 100644 ham10000/experiments/resnet18_tuned/training_log.csv

Pushed successfully. Run git pull on your laptop.


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   d54a5c6..853f093  main -> main


In [ ]:
import os, shutil
import pandas as pd

GITHUB_TOKEN = "ghp"  # paste your token
REPO         = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

# ── Clone if needed ───────────────────────────────────────────
if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system("git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git")

os.chdir(REPO)
os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')
os.system(f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git')
os.system('git pull origin main')

# ── Build complete results CSV ────────────────────────────────
results = [
    # Week 3 — baseline experiments
    {
        "experiment":          "baseline_image_only",
        "week":                3,
        "architecture":        "ResNet-18",
        "optimizer":           "AdamW",
        "lr":                  1e-4,
        "weight_decay":        1e-4,
        "loss":                "cross_entropy",
        "label_smoothing":     0.0,
        "dropout":             0.3,
        "epochs":              20,
        "best_val_epoch":      15,
        "best_val_bal_acc":    0.7964,
        "test_bal_acc":        0.7318,
        "test_macro_f1":       0.6869,
        "test_macro_auc":      0.9478,
        "notes":               "Original baseline — best model overall",
    },
    {
        "experiment":          "metadata_fusion",
        "week":                3,
        "architecture":        "ResNet-18 + metadata",
        "optimizer":           "AdamW",
        "lr":                  1e-4,
        "weight_decay":        1e-4,
        "loss":                "cross_entropy",
        "label_smoothing":     0.0,
        "dropout":             0.3,
        "epochs":              20,
        "best_val_epoch":      13,
        "best_val_bal_acc":    0.7825,
        "test_bal_acc":        0.7168,
        "test_macro_f1":       0.6703,
        "test_macro_auc":      0.9549,
        "notes":               "Late fusion 18-dim metadata vector",
    },
    # Kaggle GPU — EfficientNet experiments
    {
        "experiment":          "efficientnet_b0_v2",
        "week":                3,
        "architecture":        "EfficientNet-B0",
        "optimizer":           "AdamW",
        "lr":                  1e-4,
        "weight_decay":        1e-4,
        "loss":                "label_smoothing",
        "label_smoothing":     0.1,
        "dropout":             0.4,
        "epochs":              25,
        "best_val_epoch":      18,
        "best_val_bal_acc":    0.6630,
        "test_bal_acc":        None,
        "test_macro_f1":       None,
        "test_macro_auc":      None,
        "notes":               "Underfitting — val_loss never below 1.03",
    },
    {
        "experiment":          "efficientnet_b0_v3",
        "week":                3,
        "architecture":        "EfficientNet-B0",
        "optimizer":           "AdamW",
        "lr":                  2e-4,
        "weight_decay":        1e-4,
        "loss":                "cross_entropy",
        "label_smoothing":     0.0,
        "dropout":             0.3,
        "epochs":              30,
        "best_val_epoch":      18,
        "best_val_bal_acc":    0.7165,
        "test_bal_acc":        None,
        "test_macro_f1":       None,
        "test_macro_auc":      None,
        "notes":               "Overfitting — train_loss 0.06 vs val_loss 0.74",
    },
    # Supervisor-guided tuning
    {
        "experiment":          "resnet18_tuned",
        "week":                3,
        "architecture":        "ResNet-18",
        "optimizer":           "AdamW",
        "lr":                  5e-5,
        "weight_decay":        5e-4,
        "loss":                "label_smoothing",
        "label_smoothing":     0.1,
        "dropout":             0.4,
        "epochs":              30,
        "best_val_epoch":      18,
        "best_val_bal_acc":    0.6584,
        "test_bal_acc":        None,
        "test_macro_f1":       None,
        "test_macro_auc":      None,
        "notes":               "Supervisor feedback: lower LR + stronger L2. Worse than baseline.",
    },
    {
        "experiment":          "efficientnet_b0_tuned",
        "week":                3,
        "architecture":        "EfficientNet-B0",
        "optimizer":           "AdamW",
        "lr":                  5e-5,
        "weight_decay":        5e-4,
        "loss":                "label_smoothing",
        "label_smoothing":     0.1,
        "dropout":             0.4,
        "epochs":              30,
        "best_val_epoch":      22,
        "best_val_bal_acc":    0.6481,
        "test_bal_acc":        None,
        "test_macro_f1":       None,
        "test_macro_auc":      None,
        "notes":               "Supervisor feedback: lower LR. Worse than ENv3.",
    },
    {
        "experiment":          "efficientnet_b0_v3_tuned",
        "week":                3,
        "architecture":        "EfficientNet-B0",
        "optimizer":           "AdamW",
        "lr":                  3e-5,
        "weight_decay":        1e-3,
        "loss":                "label_smoothing",
        "label_smoothing":     0.15,
        "dropout":             0.5,
        "epochs":              30,
        "best_val_epoch":      18,
        "best_val_bal_acc":    0.6170,
        "test_bal_acc":        None,
        "test_macro_f1":       None,
        "test_macro_auc":      None,
        "notes":               "Supervisor feedback: lr=3e-5 + wd=1e-3. Underfitting.",
    },
]

df = pd.DataFrame(results)

# ── Save CSV ──────────────────────────────────────────────────
os.makedirs("ham10000/results", exist_ok=True)
csv_path = "ham10000/results/all_experiments_results.csv"
df.to_csv(csv_path, index=False)
print(f"CSV saved: {csv_path}")
print(f"Shape: {df.shape}")
print("\nSummary:")
print(df[["experiment","architecture","lr","best_val_bal_acc",
          "test_bal_acc","notes"]].to_string(index=False))

# ── Push to GitHub ────────────────────────────────────────────
os.system(f"git add {csv_path}")
os.system("""git commit -m "Add complete experiment results CSV

all_experiments_results.csv covers all Week 3 experiments:
  - baseline_image_only:      test bal-acc 0.7318 (BEST)
  - metadata_fusion:          test bal-acc 0.7168
  - efficientnet_b0_v2:       val  bal-acc 0.6630
  - efficientnet_b0_v3:       val  bal-acc 0.7165
  - resnet18_tuned:           val  bal-acc 0.6584
  - efficientnet_b0_tuned:    val  bal-acc 0.6481
  - efficientnet_b0_v3_tuned: val  bal-acc 0.6170

Conclusion: original ResNet-18 baseline remains best model.
Proceeding to Week 4 XAI implementation." """)

result = os.system("git push origin main")
if result == 0:
    print("\nPushed to GitHub successfully.")
else:
    print("\nPush failed — check token.")

From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
CSV saved: ham10000/results/all_experiments_results.csv
Shape: (7, 16)

Summary:
              experiment         architecture      lr  best_val_bal_acc  test_bal_acc                                                             notes
     baseline_image_only            ResNet-18 0.00010            0.7964        0.7318                            Original baseline — best model overall
         metadata_fusion ResNet-18 + metadata 0.00010            0.7825        0.7168                                Late fusion 18-dim metadata vector
      efficientnet_b0_v2      EfficientNet-B0 0.00010            0.6630           NaN                          Underfitting — val_loss never below 1.03
      efficientnet_b0_v3      EfficientNet-B0 0.00020            0.7165           NaN                    Overfitting — train_loss 0.06 vs val_loss 0.74
          resnet18_tuned            ResNet-18 0.00005            0.6584           NaN Supervisor feedback: lower LR + stronger L2. Worse th

To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   853f093..4c37919  main -> main


In [14]:
import os
import shutil

WORK = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/data"

if os.path.exists(WORK):
    shutil.rmtree(WORK, ignore_errors=True)

os.makedirs(WORK, exist_ok=True)

print("Clean data directory created.")

Clean data directory created.


In [15]:
import os
import shutil
import yaml
import sys

# ============================================================
# Clone repository (only if it doesn't already exist)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    !git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
else:
    print("Repository already exists.")

# ============================================================
# Dataset paths
# ============================================================

KAGGLE_DATA = "/kaggle/input/datasets/asifaliansaree/ham10000-images/data"

WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"
CHECKPOINT_DIR = f"{REPO}/ham10000/checkpoints"
EXPERIMENT_DIR = f"{REPO}/experiments"

os.makedirs(WORK, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(EXPERIMENT_DIR, exist_ok=True)

# ============================================================
# Copy dataset into repository
# ============================================================

print("Copying dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.isdir(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)

print("Dataset copied successfully.")

# ============================================================
# Python imports
# ============================================================

sys.path.insert(0, f"{REPO}/ham10000")
sys.path.insert(0, f"{REPO}/ham10000/src")

# ============================================================
# Create Kaggle configuration files
# ============================================================

experiments = {
    "kaggle_resnet18_best": "resnet18",
    "kaggle_efficientnet_b0_best": "efficientnet_b0",
}

for exp_name, architecture in experiments.items():

    ckpt_dir = f"{CHECKPOINT_DIR}/{exp_name}"
    exp_dir = f"{EXPERIMENT_DIR}/{exp_name}"

    os.makedirs(ckpt_dir, exist_ok=True)
    os.makedirs(exp_dir, exist_ok=True)

    cfg = {
        "seed": 42,

        "data": {
            "data_dir": WORK,
            "num_workers": 2,
            "img_size": 224,
            "augment": True,
        },

        "model": {
            "architecture": architecture,
            "num_classes": 7,
            "metadata_dim": 0,
            "pretrained": True,
            "dropout": 0.3,
            "freeze_epochs": 0,
        },

        "train": {
            "epochs": 35,
            "batch_size": 32,
            "lr": 1e-4,
            "weight_decay": 1e-4,
            "scheduler": "cosine",
            "gradient_accumulation": 1,
            "max_grad_norm": 1.0,
            "use_ema": False,
            "early_stopping_patience": 35,
        },

        "loss": {
            "name": "cross_entropy"
        },

        "output": {
            "checkpoint_dir": ckpt_dir
        },

        "logging": {
            "experiment_name": exp_name,
            "save_dir": exp_dir,
            "checkpoint_dir": ckpt_dir,
        },
    }

    config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

    with open(config_path, "w") as f:
        yaml.dump(cfg, f)

    print(f"Created: {config_path}")

print("\nVerifying dataset contents:")
print(os.listdir(WORK))

print("\nSetup completed successfully.")

Repository already exists.
Copying dataset...
Dataset copied successfully.
Created: /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/kaggle_resnet18_best.yaml
Created: /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/kaggle_efficientnet_b0_best.yaml

Verifying dataset contents:
['HAM10000_metadata.csv', 'HAM10000_images_part_1', 'HAM10000_images_part_2', 'class_weights.npy', 'HAM10000_split.csv']

Setup completed successfully.


In [16]:
REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

for exp in ["kaggle_resnet18_best", "kaggle_efficientnet_b0_best"]:

    print("=" * 60)
    print(f"TRAINING: {exp}")
    print("=" * 60)

    !python {REPO}/ham10000/src/train.py \
        --config {REPO}/ham10000/configs/{exp}.yaml

TRAINING: kaggle_resnet18_best

Device      : cuda
Experiment  : kaggle_resnet18_best
Architecture: resnet18
Metadata    : False
EMA         : False
Freeze ep   : 0
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 175MB/s]
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:252: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 35 epochs — loss=cross_entropy, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:166: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autoca

In [ ]:
import os, shutil, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

GITHUB_TOKEN = "ghp"  # paste your token
REPO         = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system("git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git")

os.chdir(REPO)
os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')
os.system(f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git')
os.system('git pull origin main')

# ── Reconstruct all training logs ─────────────────────────────
os.makedirs("ham10000/experiments/resnet18_best",        exist_ok=True)
os.makedirs("ham10000/experiments/efficientnet_b0_best", exist_ok=True)
os.makedirs("ham10000/results/figures",                  exist_ok=True)

resnet_best = pd.DataFrame({
    "epoch":       list(range(1, 36)),
    "train_loss":  [0.7742,0.5970,0.5443,0.4907,0.4796,0.4449,0.4233,
                    0.3925,0.3824,0.3568,0.3422,0.3279,0.3001,0.2987,
                    0.2682,0.2624,0.2524,0.2239,0.2126,0.1967,0.1909,
                    0.1814,0.1655,0.1530,0.1429,0.1273,0.1189,0.1164,
                    0.1052,0.1093,0.1092,0.1035,0.0966,0.0999,0.0930],
    "val_loss":    [0.6090,0.6490,0.5548,0.5608,0.6473,0.5445,0.5335,
                    0.6042,0.5004,0.5366,0.5814,0.4883,0.5437,0.5832,
                    0.5635,0.5450,0.6318,0.5799,0.6046,0.6144,0.6981,
                    0.6454,0.6589,0.6193,0.6301,0.6132,0.6291,0.6421,
                    0.6299,0.6582,0.6718,0.6740,0.6891,0.6849,0.6780],
    "val_bal_acc": [0.5271,0.5272,0.6167,0.5707,0.5624,0.6029,0.6240,
                    0.6209,0.6226,0.6121,0.5815,0.6349,0.6566,0.6430,
                    0.6533,0.6511,0.6380,0.6643,0.6504,0.6762,0.6512,
                    0.6542,0.6553,0.6862,0.6571,0.6891,0.6931,0.6779,
                    0.6939,0.7001,0.6919,0.6924,0.6967,0.6947,0.7082],
})

en_best = pd.DataFrame({
    "epoch":       list(range(1, 36)),
    "train_loss":  [0.9444,0.5875,0.5084,0.4676,0.4251,0.3853,0.3616,
                    0.3319,0.3161,0.2992,0.2944,0.2684,0.2486,0.2352,
                    0.2183,0.2146,0.1929,0.1687,0.1813,0.1625,0.1484,
                    0.1388,0.1426,0.1436,0.1225,0.1157,0.1241,0.1155,
                    0.1147,0.1025,0.1074,0.1044,0.1000,0.1002,0.1057],
    "val_loss":    [0.6795,0.6032,0.6061,0.5490,0.5847,0.5122,0.5132,
                    0.5091,0.5320,0.4928,0.4936,0.5105,0.5101,0.5315,
                    0.5080,0.5476,0.5475,0.5501,0.5469,0.5467,0.5421,
                    0.5496,0.5588,0.5799,0.6130,0.5655,0.5730,0.5802,
                    0.5799,0.5912,0.5962,0.5889,0.5866,0.6015,0.5820],
    "val_bal_acc": [0.4705,0.5506,0.5544,0.5715,0.5946,0.6209,0.6654,
                    0.6511,0.6391,0.6592,0.6536,0.6384,0.6564,0.6445,
                    0.6747,0.6651,0.6467,0.6626,0.6579,0.6591,0.6782,
                    0.6706,0.6658,0.6616,0.6674,0.6703,0.6636,0.6759,
                    0.6790,0.6708,0.6708,0.6692,0.6738,0.6762,0.6731],
})

resnet_best.to_csv("ham10000/experiments/resnet18_best/training_log.csv",
                   index=False)
en_best.to_csv("ham10000/experiments/efficientnet_b0_best/training_log.csv",
               index=False)
print("Training logs saved")

# ── ALL experiments summary ───────────────────────────────────
all_experiments = {
    "ResNet-18\nbaseline":       {"best": 0.7964, "ep": 15,
                                  "color": "#2a78d6", "marker": "★"},
    "ResNet-18\ntuned":          {"best": 0.6584, "ep": 18,
                                  "color": "#5ba4e0", "marker": "o"},
    "ResNet-18\nbest (35ep)":    {"best": 0.7082, "ep": 35,
                                  "color": "#1a5fa8", "marker": "o"},
    "ENv2\nlr=1e-4":             {"best": 0.6630, "ep": 18,
                                  "color": "#1baf7a", "marker": "o"},
    "ENv3\nlr=2e-4":             {"best": 0.7165, "ep": 18,
                                  "color": "#138a5e", "marker": "o"},
    "EN tuned\nlr=5e-5":         {"best": 0.6481, "ep": 22,
                                  "color": "#0d6644", "marker": "o"},
    "ENv3 tuned\nlr=3e-5":       {"best": 0.6170, "ep": 18,
                                  "color": "#08442d", "marker": "o"},
    "EN best\n(35ep)":           {"best": 0.6790, "ep": 29,
                                  "color": "#23d48f", "marker": "o"},
    "Metadata\nfusion":          {"best": 0.7825, "ep": 13,
                                  "color": "#e34948", "marker": "D"},
}

# ── Figure 1: All experiments bar chart ───────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
names  = list(all_experiments.keys())
values = [v["best"] for v in all_experiments.values()]
colors = [v["color"] for v in all_experiments.values()]

bars = ax.bar(names, values, color=colors,
              edgecolor="none", width=0.6)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f"{val:.4f}",
            ha="center", va="bottom",
            fontsize=8, fontweight="bold")

ax.axhline(0.7964, color="#2a78d6", linestyle="--",
           linewidth=1.5, alpha=0.8,
           label="Original baseline (0.7964) — best")
ax.axhline(0.65, color="#888780", linestyle=":",
           linewidth=1, alpha=0.6,
           label="Go/no-go threshold (0.65)")
ax.set_ylim(0.5, 0.90)
ax.set_ylabel("Best val balanced accuracy", fontsize=12)
ax.set_title("All experiments — val balanced accuracy comparison",
             fontsize=13)
ax.legend(fontsize=9)
ax.grid(axis="y", linewidth=0.4, alpha=0.5)
fig.tight_layout()
fig.savefig("ham10000/results/figures/all_experiments_bar.png",
            dpi=300, bbox_inches="tight")
plt.close()
print("Saved: all_experiments_bar.png")

# ── Figure 2: ResNet-18 variants learning curves ──────────────
original_resnet = pd.DataFrame({
    "epoch":       list(range(1, 21)),
    "train_loss":  [1.2297,0.9080,0.7725,0.7301,0.6584,0.5955,0.5583,
                    0.5146,0.4565,0.4230,0.4007,0.3679,0.3119,0.2905,
                    0.2819,0.2478,0.2406,0.2236,0.2179,0.2238],
    "val_loss":    [0.9273,0.9599,0.8957,0.7727,0.7851,0.8813,0.7096,
                    0.7006,0.7027,0.6521,0.6466,0.6169,0.6048,0.6158,
                    0.6083,0.6050,0.6350,0.6180,0.5847,0.6452],
    "val_bal_acc": [0.6486,0.6882,0.6880,0.7617,0.7087,0.7156,0.7110,
                    0.7337,0.7460,0.7798,0.7561,0.7587,0.7848,0.7714,
                    0.7964,0.7758,0.7705,0.7882,0.7902,0.7843],
})

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
datasets = [
    ("ResNet-18 original\n(lr=1e-4, 20ep, best=0.7964)",
     original_resnet, "#2a78d6"),
    ("ResNet-18 tuned\n(lr=5e-5, 30ep, best=0.6584)",
     pd.DataFrame({
         "epoch":       list(range(1,31)),
         "train_loss":  [1.2507,1.0857,1.0308,0.9820,0.9642,0.9387,
                         0.9322,0.9003,0.8903,0.8794,0.8711,0.8539,
                         0.8398,0.8295,0.8135,0.8098,0.8044,0.7892,
                         0.7826,0.7730,0.7673,0.7681,0.7589,0.7541,
                         0.7533,0.7468,0.7356,0.7425,0.7332,0.7397],
         "val_bal_acc": [0.4818,0.5421,0.5937,0.5061,0.5735,0.5772,
                         0.6227,0.6073,0.6313,0.6426,0.5901,0.6185,
                         0.6141,0.6024,0.6180,0.6288,0.6304,0.6584,
                         0.6543,0.6570,0.6334,0.6403,0.6407,0.6537,
                         0.6442,0.6423,0.6577,0.6322,0.6463,0.6422],
     }), "#5ba4e0"),
    ("ResNet-18 best\n(lr=1e-4, 35ep, best=0.7082)",
     resnet_best, "#1a5fa8"),
]

for ax, (title, df, color) in zip(axes, datasets):
    ax2 = ax.twinx()
    ax.plot(df["epoch"], df["train_loss"],
            color="#e34948", linewidth=2,
            linestyle="--", label="train loss")
    if "val_loss" in df.columns:
        ax.plot(df["epoch"], df["val_loss"],
                color="#EF9F27", linewidth=2,
                label="val loss")
    ax2.plot(df["epoch"], df["val_bal_acc"],
             color=color, linewidth=2.5,
             label="val bal-acc")
    best     = df["val_bal_acc"].max()
    best_ep  = int(df.loc[df["val_bal_acc"].idxmax(), "epoch"])
    ax2.axvline(best_ep, color="#888780",
                linestyle=":", linewidth=1.2)
    ax.set_xlabel("Epoch", fontsize=10)
    ax.set_ylabel("Loss", fontsize=9, color="#888780")
    ax2.set_ylabel("Bal acc", fontsize=9, color=color)
    ax.set_title(title, fontsize=9)
    ax.grid(linewidth=0.4, alpha=0.4)
    lines1, lbl1 = ax.get_legend_handles_labels()
    lines2, lbl2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, lbl1+lbl2, fontsize=7, loc="upper left")

fig.suptitle("ResNet-18 variants — training curves comparison",
             fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig("ham10000/results/figures/resnet18_variants_curves.png",
            dpi=300, bbox_inches="tight")
plt.close()
print("Saved: resnet18_variants_curves.png")

# ── Figure 3: EfficientNet variants ───────────────────────────
en_v2 = pd.DataFrame({
    "epoch":       list(range(1,26)),
    "train_loss":  [1.3154,1.0644,1.0106,0.9804,0.9540,0.9310,0.9087,
                    0.8945,0.8867,0.8743,0.8588,0.8555,0.8435,0.8255,
                    0.8208,0.8062,0.7973,0.7895,0.7922,0.7782,0.7814,
                    0.7769,0.7719,0.7714,0.7781],
    "val_bal_acc": [0.4478,0.5128,0.5357,0.5598,0.6140,0.6064,0.6352,
                    0.6202,0.6145,0.6336,0.6179,0.6370,0.6347,0.6308,
                    0.6287,0.6431,0.6336,0.6630,0.6536,0.6378,0.6324,
                    0.6423,0.6338,0.6359,0.6535],
})
en_v3 = pd.DataFrame({
    "epoch":       list(range(1,31)),
    "train_loss":  [0.8135,0.5623,0.4891,0.4437,0.4012,0.3720,0.3466,
                    0.3114,0.2804,0.2641,0.2497,0.2230,0.1973,0.1867,
                    0.1663,0.1572,0.1368,0.1144,0.1269,0.1076,0.0990,
                    0.0862,0.0807,0.0790,0.0746,0.0616,0.0695,0.0575,
                    0.0660,0.0571],
    "val_bal_acc": [0.5121,0.5815,0.5897,0.5843,0.6068,0.6357,0.6627,
                    0.6623,0.6617,0.6834,0.6758,0.6765,0.6937,0.6754,
                    0.7058,0.6991,0.6851,0.7165,0.6992,0.6823,0.7026,
                    0.6999,0.7050,0.6925,0.7149,0.7088,0.7024,0.7095,
                    0.7035,0.7068],
})

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(en_v2["epoch"], en_v2["val_bal_acc"],
        color="#1baf7a", linewidth=2,
        label="ENv2 (lr=1e-4, 25ep) best=0.6630")
ax.plot(en_v3["epoch"], en_v3["val_bal_acc"],
        color="#138a5e", linewidth=2,
        label="ENv3 (lr=2e-4, 30ep) best=0.7165")
ax.plot(en_best["epoch"], en_best["val_bal_acc"],
        color="#23d48f", linewidth=2,
        label="EN best (lr=1e-4, 35ep) best=0.6790")

for name, df, color in [
    ("ENv2",      en_v2,    "#1baf7a"),
    ("ENv3",      en_v3,    "#138a5e"),
    ("EN best",   en_best,  "#23d48f"),
]:
    best     = df["val_bal_acc"].max()
    best_ep  = int(df.loc[df["val_bal_acc"].idxmax(), "epoch"])
    ax.scatter(best_ep, best, color=color, s=80, zorder=5)
    ax.annotate(f"best={best:.4f}",
                xy=(best_ep, best),
                xytext=(best_ep+0.5, best+0.008),
                fontsize=8, color=color)

ax.axhline(0.7964, color="#2a78d6", linestyle="--",
           linewidth=1.5, alpha=0.8,
           label="ResNet-18 baseline (0.7964)")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Val balanced accuracy", fontsize=12)
ax.set_title("EfficientNet-B0 variants — val balanced accuracy",
             fontsize=13)
ax.legend(fontsize=9)
ax.grid(linewidth=0.4, alpha=0.5)
ax.set_ylim(0.40, 0.85)
fig.tight_layout()
fig.savefig("ham10000/results/figures/efficientnet_variants_curves.png",
            dpi=300, bbox_inches="tight")
plt.close()
print("Saved: efficientnet_variants_curves.png")

# ── Update results CSV ────────────────────────────────────────
import pandas as pd

csv_path = "ham10000/results/all_experiments_results.csv"
if os.path.exists(csv_path):
    df_results = pd.read_csv(csv_path)
else:
    df_results = pd.DataFrame()

new_rows = pd.DataFrame([
    {
        "experiment":       "resnet18_best",
        "week":             3,
        "architecture":     "ResNet-18",
        "optimizer":        "AdamW",
        "lr":               1e-4,
        "weight_decay":     1e-4,
        "loss":             "cross_entropy",
        "label_smoothing":  0.0,
        "dropout":          0.3,
        "epochs":           35,
        "best_val_epoch":   35,
        "best_val_bal_acc": 0.7082,
        "test_bal_acc":     None,
        "test_macro_f1":    None,
        "test_macro_auc":   None,
        "notes":            "35-epoch run — augmentation change hurt vs original",
    },
    {
        "experiment":       "efficientnet_b0_best",
        "week":             3,
        "architecture":     "EfficientNet-B0",
        "optimizer":        "AdamW",
        "lr":               1e-4,
        "weight_decay":     1e-4,
        "loss":             "cross_entropy",
        "label_smoothing":  0.0,
        "dropout":          0.3,
        "epochs":           35,
        "best_val_epoch":   29,
        "best_val_bal_acc": 0.6790,
        "test_bal_acc":     None,
        "test_macro_f1":    None,
        "test_macro_auc":   None,
        "notes":            "35-epoch run — consistent with previous EN runs",
    },
])

df_results = pd.concat([df_results, new_rows], ignore_index=True)
df_results.to_csv(csv_path, index=False)
print(f"Updated: {csv_path}")

# ── Push everything ───────────────────────────────────────────
os.system("git add ham10000/results/figures/all_experiments_bar.png")
os.system("git add ham10000/results/figures/resnet18_variants_curves.png")
os.system("git add ham10000/results/figures/efficientnet_variants_curves.png")
os.system("git add ham10000/results/all_experiments_results.csv")
os.system("git add ham10000/experiments/resnet18_best/")
os.system("git add ham10000/experiments/efficientnet_b0_best/")

os.system("""git commit -m "Add 35-epoch experiment results + 3 comparison figures

resnet18_best (35ep):        best val bal-acc 0.7082 ep35
efficientnet_b0_best (35ep): best val bal-acc 0.6790 ep29

Figures added:
- all_experiments_bar.png: all 9 experiments compared
- resnet18_variants_curves.png: 3 ResNet-18 variants
- efficientnet_variants_curves.png: 3 EfficientNet variants

FINAL CONCLUSION: original ResNet-18 baseline (0.7964) is best.
Augmentation change in resnet18_best caused regression.
All training experiments complete. Proceeding to Week 4 XAI." """)

result = os.system("git push origin main")
if result == 0:
    print("\nAll pushed successfully.")
    print("Run: git pull on your laptop")
else:
    print("\nPush failed — check token")

From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Training logs saved
Saved: all_experiments_bar.png
Saved: resnet18_variants_curves.png
Saved: efficientnet_variants_curves.png
Updated: ham10000/results/all_experiments_results.csv
[main bf75ae3] Add 35-epoch experiment results + 3 comparison figures
 1 file changed, 2 insertions(+)


/tmp/ipykernel_58/1956615274.py:307: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, new_rows], ignore_index=True)



All pushed successfully.
Run: git pull on your laptop


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   ddf4f79..bf75ae3  main -> main


In [4]:
import os, getpass, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Paste your token when prompted -- do NOT hardcode it in the cell.
GITHUB_TOKEN = getpass.getpass("GitHub token: ")
REPO         = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system("git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git")

os.chdir(REPO)
os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')
os.system(f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git')
os.system('git pull origin main')

# ── Reconstruct training logs for the 3 "tuned" runs ──────────
os.makedirs("ham10000/experiments/resnet18_tuned",           exist_ok=True)
os.makedirs("ham10000/experiments/efficientnet_b0_tuned",    exist_ok=True)
os.makedirs("ham10000/experiments/efficientnet_b0_v3_tuned", exist_ok=True)
os.makedirs("ham10000/results/figures",                      exist_ok=True)

resnet18_tuned = pd.DataFrame({
    "epoch":       list(range(1, 31)),
    "train_loss":  [1.2507,1.0857,1.0308,0.9820,0.9642,0.9387,0.9322,0.9003,0.8903,0.8794,
                    0.8711,0.8539,0.8398,0.8295,0.8135,0.8098,0.8044,0.7892,0.7826,0.7730,
                    0.7673,0.7681,0.7589,0.7541,0.7533,0.7468,0.7356,0.7425,0.7332,0.7397],
    "val_loss":    [1.1587,1.1424,1.0860,1.1092,1.1078,1.0805,1.0499,1.0923,1.0368,1.0431,
                    1.0501,1.0550,1.0810,1.1096,1.0589,1.0370,1.0505,1.0606,1.0490,1.0502,
                    1.0771,1.0611,1.0556,1.0711,1.0637,1.0457,1.0483,1.0520,1.0534,1.0529],
    "val_bal_acc": [0.4818,0.5421,0.5937,0.5061,0.5735,0.5772,0.6227,0.6073,0.6313,0.6426,
                    0.5901,0.6185,0.6141,0.6024,0.6180,0.6288,0.6304,0.6584,0.6543,0.6570,
                    0.6334,0.6403,0.6407,0.6537,0.6442,0.6423,0.6577,0.6322,0.6463,0.6422],
    "lr":          [4.99e-05,4.95e-05,4.88e-05,4.79e-05,4.67e-05,4.53e-05,4.37e-05,4.19e-05,3.99e-05,3.78e-05,
                    3.55e-05,3.31e-05,3.06e-05,2.81e-05,2.55e-05,2.29e-05,2.04e-05,1.79e-05,1.55e-05,1.33e-05,
                    1.11e-05,9.11e-06,7.29e-06,5.68e-06,4.28e-06,3.12e-06,2.20e-06,1.54e-06,1.13e-06,1.00e-06],
})

efficientnet_b0_tuned = pd.DataFrame({
    "epoch":       list(range(1, 31)),
    "train_loss":  [1.4399,1.0948,1.0182,0.9838,0.9535,0.9272,0.9134,0.8984,0.8824,0.8761,
                    0.8686,0.8538,0.8471,0.8313,0.8352,0.8134,0.8101,0.7989,0.8073,0.7991,
                    0.7886,0.7933,0.7881,0.7871,0.7811,0.7782,0.7856,0.7754,0.7767,0.7735],
    "val_loss":    [1.2724,1.1684,1.1317,1.0905,1.1006,1.0760,1.0683,1.0826,1.0759,1.0451,
                    1.0522,1.0799,1.0442,1.0284,1.0463,1.0419,1.0375,1.0348,1.0531,1.0558,
                    1.0535,1.0404,1.0525,1.0519,1.0607,1.0446,1.0512,1.0514,1.0451,1.0431],
    "val_bal_acc": [0.4266,0.4799,0.5001,0.5481,0.5643,0.5752,0.6160,0.6016,0.5873,0.6098,
                    0.6054,0.6037,0.6299,0.6371,0.6169,0.6440,0.6318,0.6449,0.6191,0.6156,
                    0.6140,0.6481,0.6267,0.6359,0.6249,0.6411,0.6428,0.6184,0.6253,0.6386],
    "lr":          [4.99e-05,4.95e-05,4.88e-05,4.79e-05,4.67e-05,4.53e-05,4.37e-05,4.19e-05,3.99e-05,3.78e-05,
                    3.55e-05,3.31e-05,3.06e-05,2.81e-05,2.55e-05,2.29e-05,2.04e-05,1.79e-05,1.55e-05,1.33e-05,
                    1.11e-05,9.11e-06,7.29e-06,5.68e-06,4.28e-06,3.12e-06,2.20e-06,1.54e-06,1.13e-06,1.00e-06],
})

efficientnet_b0_v3_tuned = pd.DataFrame({
    "epoch":       list(range(1, 31)),
    "train_loss":  [1.7053,1.3345,1.2428,1.2021,1.1721,1.1467,1.1364,1.1215,1.1052,1.1035,
                    1.0952,1.0895,1.0807,1.0667,1.0703,1.0529,1.0505,1.0405,1.0489,1.0428,
                    1.0379,1.0397,1.0368,1.0362,1.0284,1.0274,1.0325,1.0256,1.0312,1.0228],
    "val_loss":    [1.5663,1.4136,1.3567,1.3247,1.3119,1.2950,1.2861,1.2911,1.2787,1.2680,
                    1.2626,1.2749,1.2505,1.2445,1.2500,1.2494,1.2479,1.2535,1.2621,1.2626,
                    1.2567,1.2502,1.2609,1.2603,1.2672,1.2567,1.2558,1.2642,1.2557,1.2500],
    "val_bal_acc": [0.3579,0.4453,0.4747,0.4886,0.5117,0.5228,0.5695,0.5547,0.5553,0.5679,
                    0.5804,0.5872,0.6022,0.6156,0.5915,0.6044,0.5933,0.6170,0.5976,0.5896,
                    0.5947,0.6035,0.5849,0.5900,0.5984,0.5956,0.6002,0.5839,0.5835,0.5976],
    "lr":          [2.99e-05,2.97e-05,2.93e-05,2.87e-05,2.81e-05,2.72e-05,2.63e-05,2.52e-05,2.40e-05,2.27e-05,
                    2.14e-05,2.00e-05,1.85e-05,1.70e-05,1.55e-05,1.40e-05,1.25e-05,1.10e-05,9.60e-06,8.25e-06,
                    6.98e-06,5.80e-06,4.72e-06,3.77e-06,2.94e-06,2.25e-06,1.71e-06,1.32e-06,1.08e-06,1.00e-06],
})

resnet18_tuned.to_csv("ham10000/experiments/resnet18_tuned/training_log.csv", index=False)
efficientnet_b0_tuned.to_csv("ham10000/experiments/efficientnet_b0_tuned/training_log.csv", index=False)
efficientnet_b0_v3_tuned.to_csv("ham10000/experiments/efficientnet_b0_v3_tuned/training_log.csv", index=False)
print("Training logs saved")

# ── Comparison figure: all 3 tuned runs ────────────────────────
runs = [
    ("ResNet-18 tuned\n(lr=5e-5, 30ep, best=0.6584 ep18)",       resnet18_tuned,           "#5ba4e0"),
    ("EfficientNet-B0 tuned\n(lr=5e-5, 30ep, best=0.6481 ep22)", efficientnet_b0_tuned,    "#1baf7a"),
    ("EfficientNet-B0 v3 tuned\n(lr=3e-5, 30ep, best=0.6170 ep18)", efficientnet_b0_v3_tuned, "#08442d"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (title, df, color) in zip(axes, runs):
    ax2 = ax.twinx()
    ax.plot(df["epoch"], df["train_loss"], color="#e34948", linewidth=2, linestyle="--", label="train loss")
    ax.plot(df["epoch"], df["val_loss"],   color="#EF9F27", linewidth=2, label="val loss")
    ax2.plot(df["epoch"], df["val_bal_acc"], color=color, linewidth=2.5, label="val bal-acc")
    best_ep = int(df.loc[df["val_bal_acc"].idxmax(), "epoch"])
    ax2.axvline(best_ep, color="#888780", linestyle=":", linewidth=1.2)
    ax.set_xlabel("Epoch", fontsize=10)
    ax.set_ylabel("Loss", fontsize=9, color="#888780")
    ax2.set_ylabel("Bal acc", fontsize=9, color=color)
    ax.set_title(title, fontsize=9)
    ax.grid(linewidth=0.4, alpha=0.4)
    l1, lb1 = ax.get_legend_handles_labels()
    l2, lb2 = ax2.get_legend_handles_labels()
    ax.legend(l1+l2, lb1+lb2, fontsize=7, loc="upper left")

fig.suptitle("Tuned-run comparison — lower LR did not beat the original baseline", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig("ham10000/results/figures/tuned_runs_curves.png", dpi=300, bbox_inches="tight")
plt.close()
print("Saved: tuned_runs_curves.png")

# ── Append to results CSV ──────────────────────────────────────
csv_path = "ham10000/results/all_experiments_results.csv"
df_results = pd.read_csv(csv_path) if os.path.exists(csv_path) else pd.DataFrame()

new_rows = pd.DataFrame([
    {"experiment":"resnet18_tuned","week":3,"architecture":"ResNet-18","optimizer":"AdamW",
     "lr":5e-5,"weight_decay":None,"loss":"label_smoothing","label_smoothing":None,"dropout":None,
     "epochs":30,"best_val_epoch":18,"best_val_bal_acc":0.6584,
     "test_bal_acc":None,"test_macro_f1":None,"test_macro_auc":None,
     "notes":"Lower LR (5e-5) + label smoothing underperformed original baseline (0.7964)"},
    {"experiment":"efficientnet_b0_tuned","week":3,"architecture":"EfficientNet-B0","optimizer":"AdamW",
     "lr":5e-5,"weight_decay":None,"loss":"label_smoothing","label_smoothing":None,"dropout":None,
     "epochs":30,"best_val_epoch":22,"best_val_bal_acc":0.6481,
     "test_bal_acc":None,"test_macro_f1":None,"test_macro_auc":None,
     "notes":"Consistent underperformance vs ResNet-18 baseline"},
    {"experiment":"efficientnet_b0_v3_tuned","week":3,"architecture":"EfficientNet-B0","optimizer":"AdamW",
     "lr":3e-5,"weight_decay":None,"loss":"label_smoothing","label_smoothing":None,"dropout":None,
     "epochs":30,"best_val_epoch":18,"best_val_bal_acc":0.6170,
     "test_bal_acc":None,"test_macro_f1":None,"test_macro_auc":None,
     "notes":"Lowest LR variant — slowest convergence, weakest result"},
])

df_results = pd.concat([df_results, new_rows], ignore_index=True)
df_results.to_csv(csv_path, index=False)
print(f"Updated: {csv_path}")

# ── Push ────────────────────────────────────────────────────
os.system("git add ham10000/results/figures/tuned_runs_curves.png")
os.system("git add ham10000/results/all_experiments_results.csv")
os.system("git add ham10000/experiments/resnet18_tuned/")
os.system("git add ham10000/experiments/efficientnet_b0_tuned/")
os.system("git add ham10000/experiments/efficientnet_b0_v3_tuned/")

os.system("""git commit -m "Add tuned-LR experiment results + comparison figure

resnet18_tuned:           best val bal-acc 0.6584 ep18 (lr=5e-5)
efficientnet_b0_tuned:    best val bal-acc 0.6481 ep22 (lr=5e-5)
efficientnet_b0_v3_tuned: best val bal-acc 0.6170 ep18 (lr=3e-5)

All three underperform the original ResNet-18 baseline (0.7964).
Figure added: tuned_runs_curves.png" """)

result = os.system("git push origin main")
if result == 0:
    print("\nAll pushed successfully.")
else:
    print("\nPush failed — check token")

GitHub token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Training logs saved
Saved: tuned_runs_curves.png
Updated: ham10000/results/all_experiments_results.csv
[main ea86c4e] Add tuned-LR experiment results + comparison figure
 5 files changed, 96 insertions(+), 93 deletions(-)
 rewrite ham10000/experiments/efficientnet_b0_tuned/training_log.csv (100%)
 rewrite ham10000/experiments/efficientnet_b0_v3_tuned/training_log.csv (100%)
 rewrite ham10000/experiments/resnet18_tuned/training_log.csv (100%)
 create mode 100644 ham10000/results/figures/tuned_runs_curves.png


/tmp/ipykernel_58/350088388.py:131: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, new_rows], ignore_index=True)



All pushed successfully.


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   bf75ae3..ea86c4e  main -> main


In [27]:
%cd /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging
!git pull origin main

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 7 (delta 6), reused 7 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 590 bytes | 295.00 KiB/s, done.
From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD
   8dee516..4ec7caa  main       -> origin/main
Updating 8dee516..4ec7caa
Fast-forward
 ham10000/configs/ablation_focal.yaml    | 1 +
 ham10000/configs/ablation_onecycle.yaml | 1 +
 ham10000/configs/ablation_sampler.yaml  | 1 +
 3 files changed, 3 insertions(+)


In [3]:
import os
import shutil
import yaml
import sys

# ============================================================
# Clone repository (only if it doesn't already exist)
# ============================================================
REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
if not os.path.exists(REPO):
    !git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
else:
    print("Repository already exists.")

# ============================================================
# Dataset paths
# ============================================================
KAGGLE_DATA     = "/kaggle/input/datasets/asifaliansaree/ham10000-images/data"
WORK            = f"{REPO}/ham10000/data"
CONFIG_DIR      = f"{REPO}/ham10000/configs"
CHECKPOINT_DIR  = f"{REPO}/ham10000/checkpoints"
EXPERIMENT_DIR  = f"{REPO}/experiments"

os.makedirs(WORK, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(EXPERIMENT_DIR, exist_ok=True)

# ============================================================
# linking dataset into repository
# ============================================================
print("Linking dataset...")
for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)
    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)
    os.symlink(src, dst)
print("Dataset linked successfully.")

# ============================================================
# Python imports
# ============================================================
sys.path.insert(0, f"{REPO}/ham10000")
sys.path.insert(0, f"{REPO}/ham10000/src")

# ============================================================
# Measured LR from lr_finder.py — update if you re-run it
# ============================================================
MEASURED_LR = 7.94e-05

# ============================================================
# Create ablation configuration files
# ============================================================
ablation_configs = {
    "ablation_onecycle": {
        "scheduler": "onecycle",
        "loss": {"name": "cross_entropy"},
        "use_weighted_sampler": False,
    },
    "ablation_focal": {
        "scheduler": "onecycle",
        "loss": {"name": "focal", "focal_gamma": 2.0},
        "use_weighted_sampler": False,
    },
    "ablation_sampler": {
        "scheduler": "onecycle",
        "loss": {"name": "cross_entropy"},
        "use_weighted_sampler": True,
    },
}

for exp_name, settings in ablation_configs.items():
    ckpt_dir = f"{CHECKPOINT_DIR}/{exp_name}"
    exp_dir  = f"{EXPERIMENT_DIR}/{exp_name}"
    os.makedirs(ckpt_dir, exist_ok=True)
    os.makedirs(exp_dir, exist_ok=True)

    cfg = {
        "seed": 42,
        "data": {
            "data_dir": WORK,
            "num_workers": 2,
            "img_size": 224,
            "augment": True,
        },
        "model": {
            "architecture": "resnet18",
            "num_classes": 7,
            "metadata_dim": 0,
            "pretrained": True,
            "dropout": 0.3,
            "freeze_epochs": 0,
        },
        "train": {
            "epochs": 20,
            "batch_size": 32,
            "lr": MEASURED_LR,
            "weight_decay": 1e-4,
            "scheduler": settings["scheduler"],
            "gradient_accumulation": 1,
            "max_grad_norm": 1.0,
            "use_ema": False,
            "early_stopping_patience": 20,
            "use_weighted_sampler": settings["use_weighted_sampler"],
        },
        "loss": settings["loss"],
        "output": {
            "checkpoint_dir": ckpt_dir
        },
        "logging": {
            "experiment_name": exp_name,
            "save_dir": exp_dir,
            "checkpoint_dir": ckpt_dir,
        },
    }

    config_path = f"{CONFIG_DIR}/{exp_name}.yaml"
    with open(config_path, "w") as f:
        yaml.dump(cfg, f)
    print(f"Created: {config_path}")

print("\nVerifying dataset contents:")
print(os.listdir(WORK))
print("\nSetup completed successfully.")

Repository already exists.
Linking dataset...
Dataset linked successfully.
Created: /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/ablation_onecycle.yaml
Created: /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/ablation_focal.yaml
Created: /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/ablation_sampler.yaml

Verifying dataset contents:
['HAM10000_images_part_2', 'HAM10000_split.csv', 'class_weights.npy', 'HAM10000_metadata.csv', 'HAM10000_images_part_1']

Setup completed successfully.


In [4]:
REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

for exp in ["ablation_onecycle", "ablation_focal", "ablation_sampler"]:
    print("=" * 60)
    print(f"TRAINING: {exp}")
    print("=" * 60)
    !python {REPO}/ham10000/src/train.py \
        --config {REPO}/ham10000/configs/{exp}.yaml

TRAINING: ablation_onecycle

Device      : cuda
Experiment  : ablation_onecycle
Architecture: resnet18
Metadata    : False
EMA         : False
Freeze ep   : 0
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 232MB/s]
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:301: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 20 epochs — loss=cross_entropy, scheduler=onecycle

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:215: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('

In [14]:
import os, getpass, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Paste your token when prompted -- do NOT hardcode it in the cell.
GITHUB_TOKEN = getpass.getpass("GitHub token: ")
REPO         = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system("git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git")

os.chdir(REPO)
os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')
os.system(f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git')
os.system('git pull origin main')

# ── Reconstruct training logs for the 3 ablation runs ──────────
os.makedirs("ham10000/experiments/ablation_onecycle", exist_ok=True)
os.makedirs("ham10000/experiments/ablation_focal",    exist_ok=True)
os.makedirs("ham10000/experiments/ablation_sampler",  exist_ok=True)
os.makedirs("ham10000/results/figures",               exist_ok=True)

ablation_onecycle = pd.DataFrame({
    "epoch":       list(range(1, 21)),
    "train_loss":  [1.6176,1.2286,1.0151,0.8929,0.8112,0.7581,0.7352,0.6886,0.6795,0.6468,
                    0.6311,0.6253,0.5977,0.6036,0.5764,0.5736,0.5629,0.5574,0.5415,0.5347],
    "val_loss":    [1.2969,1.1085,0.9349,0.8644,0.7835,0.7307,0.6991,0.6794,0.6528,0.6393,
                    0.6217,0.6194,0.6061,0.6268,0.6100,0.6193,0.5941,0.5904,0.5873,0.5971],
    "val_bal_acc": [0.3063,0.3341,0.3389,0.4211,0.4432,0.4666,0.4749,0.4776,0.4775,0.4879,
                    0.4845,0.4879,0.4885,0.5095,0.4960,0.5133,0.5048,0.5162,0.5111,0.5111],
    "lr":          [3.18e-06]*20,
})

ablation_focal = pd.DataFrame({
    "epoch":       list(range(1, 21)),
    "train_loss":  [1.0883,0.7234,0.5760,0.5040,0.4631,0.4327,0.4231,0.3872,0.3810,0.3657,
                    0.3522,0.3473,0.3366,0.3364,0.3192,0.3127,0.3023,0.3033,0.2940,0.2916],
    "val_loss":    [0.7372,0.5681,0.4764,0.4349,0.4008,0.3722,0.3567,0.3404,0.3308,0.3195,
                    0.3117,0.3065,0.3003,0.2929,0.2910,0.2823,0.2787,0.2700,0.2657,0.2661],
    "val_bal_acc": [0.2690,0.2885,0.3089,0.3554,0.3778,0.4155,0.4285,0.4704,0.4969,0.5420,
                    0.5389,0.5398,0.5216,0.5626,0.5373,0.5619,0.5531,0.5837,0.5949,0.5754],
    "lr":          [3.18e-06]*20,
})

ablation_sampler = pd.DataFrame({
    "epoch":       list(range(1, 21)),
    "train_loss":  [1.6842,1.0209,0.7882,0.7148,0.6591,0.6210,0.5817,0.5432,0.5561,0.5139,
                    0.4894,0.4702,0.4751,0.4620,0.4461,0.4496,0.4483,0.4080,0.3961,0.4077],
    "val_loss":    [1.5463,1.4632,1.2989,1.1684,1.1865,1.0944,1.0491,0.9944,1.0388,1.0207,
                    0.9791,0.9859,0.9453,0.9492,0.8980,0.8888,0.9249,0.9210,0.8921,0.8919],
    "val_bal_acc": [0.3829,0.4160,0.4492,0.4558,0.4660,0.4748,0.4858,0.4986,0.5041,0.5210,
                    0.5291,0.5351,0.5501,0.5521,0.5653,0.5715,0.5778,0.5771,0.5901,0.5934],
    "lr":          [3.18e-06]*20,
})

ablation_onecycle.to_csv("ham10000/experiments/ablation_onecycle/training_log.csv", index=False)
ablation_focal.to_csv("ham10000/experiments/ablation_focal/training_log.csv", index=False)
ablation_sampler.to_csv("ham10000/experiments/ablation_sampler/training_log.csv", index=False)
print("Training logs saved")

# ── Comparison figure: all 3 ablation runs ──────────────────────
runs = [
    ("OneCycle baseline\n(CE loss, 20ep, best=0.5162 ep18)",       ablation_onecycle, "#5ba4e0"),
    ("Focal loss\n(20ep, best=0.5949 ep19)",                        ablation_focal,    "#1baf7a"),
    ("WeightedRandomSampler\n(CE loss, 20ep, best=0.5934 ep20)",    ablation_sampler,  "#e08a1b"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (title, df, color) in zip(axes, runs):
    ax2 = ax.twinx()
    ax.plot(df["epoch"], df["train_loss"], color="#e34948", linewidth=2, linestyle="--", label="train loss")
    ax.plot(df["epoch"], df["val_loss"],   color="#EF9F27", linewidth=2, label="val loss")
    ax2.plot(df["epoch"], df["val_bal_acc"], color=color, linewidth=2.5, label="val bal-acc")
    best_ep = int(df.loc[df["val_bal_acc"].idxmax(), "epoch"])
    ax2.axvline(best_ep, color="#888780", linestyle=":", linewidth=1.2)
    ax.set_xlabel("Epoch", fontsize=10)
    ax.set_ylabel("Loss", fontsize=9, color="#888780")
    ax2.set_ylabel("Bal acc", fontsize=9, color=color)
    ax.set_title(title, fontsize=9)
    ax.grid(linewidth=0.4, alpha=0.4)
    l1, lb1 = ax.get_legend_handles_labels()
    l2, lb2 = ax2.get_legend_handles_labels()
    ax.legend(l1+l2, lb1+lb2, fontsize=7, loc="upper left")

fig.suptitle("Ablation comparison — focal loss gives the best balanced accuracy of the three", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig("ham10000/results/figures/ablation_curves.png", dpi=300, bbox_inches="tight")
plt.close()
print("Saved: ablation_curves.png")

# ── Append to results CSV ──────────────────────────────────────
csv_path = "ham10000/results/all_experiments_results.csv"
df_results = pd.read_csv(csv_path) if os.path.exists(csv_path) else pd.DataFrame()

new_rows = pd.DataFrame([
    {"experiment":"ablation_onecycle","week":3,"architecture":"ResNet-18","optimizer":"AdamW",
     "lr":3.18e-06,"weight_decay":None,"loss":"cross_entropy","label_smoothing":None,"dropout":None,
     "epochs":20,"best_val_epoch":18,"best_val_bal_acc":0.5162,
     "test_bal_acc":None,"test_macro_f1":None,"test_macro_auc":None,
     "notes":"OneCycle scheduler ablation, no EMA, no metadata fusion, no oversampling"},
    {"experiment":"ablation_focal","week":3,"architecture":"ResNet-18","optimizer":"AdamW",
     "lr":3.18e-06,"weight_decay":None,"loss":"focal","label_smoothing":None,"dropout":None,
     "epochs":20,"best_val_epoch":19,"best_val_bal_acc":0.5949,
     "test_bal_acc":None,"test_macro_f1":None,"test_macro_auc":None,
     "notes":"Focal loss ablation — best of the three ablations, but still below original baseline (0.7964)"},
    {"experiment":"ablation_sampler","week":3,"architecture":"ResNet-18","optimizer":"AdamW",
     "lr":3.18e-06,"weight_decay":None,"loss":"cross_entropy","label_smoothing":None,"dropout":None,
     "epochs":20,"best_val_epoch":20,"best_val_bal_acc":0.5934,
     "test_bal_acc":None,"test_macro_f1":None,"test_macro_auc":None,
     "notes":"WeightedRandomSampler oversampling ablation (class counts 890/5367/412/257/891/86/114); bal-acc still climbing at ep20, may benefit from more epochs"},
])

df_results = pd.concat([df_results, new_rows], ignore_index=True)
df_results.to_csv(csv_path, index=False)
print(f"Updated: {csv_path}")

# ── Push ────────────────────────────────────────────────────
os.system("git add ham10000/results/figures/ablation_curves.png")
os.system("git add ham10000/results/all_experiments_results.csv")
os.system("git add ham10000/experiments/ablation_onecycle/")
os.system("git add ham10000/experiments/ablation_focal/")
os.system("git add ham10000/experiments/ablation_sampler/")

os.system("""git commit -m "Add ablation study results (onecycle/focal/sampler) + comparison figure

ablation_onecycle: best val bal-acc 0.5162 ep18 (CE loss)
ablation_focal:    best val bal-acc 0.5949 ep19 (focal loss)
ablation_sampler:  best val bal-acc 0.5934 ep20 (WeightedRandomSampler)

Focal loss and oversampling both clearly beat plain CE, but all three
remain below the original ResNet-18 baseline (0.7964).
Figure added: ablation_curves.png" """)

result = os.system("git push origin main")
if result == 0:
    print("\nAll pushed successfully.")
else:
    print("\nPush failed — check token")

GitHub token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Training logs saved
Saved: ablation_curves.png
Updated: ham10000/results/all_experiments_results.csv


/tmp/ipykernel_58/3772347253.py:116: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, new_rows], ignore_index=True)


[main ec02282] Add ablation study results (onecycle/focal/sampler) + comparison figure
 5 files changed, 66 insertions(+)
 create mode 100644 ham10000/experiments/ablation_focal/training_log.csv
 create mode 100644 ham10000/experiments/ablation_onecycle/training_log.csv
 create mode 100644 ham10000/experiments/ablation_sampler/training_log.csv
 create mode 100644 ham10000/results/figures/ablation_curves.png

All pushed successfully.


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   4ec7caa..ec02282  main -> main


In [1]:
import os
import yaml
import getpass

# ============================================================
# Setup: repo + dataset linking (safe to re-run, skips if done)
# ============================================================
REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system("git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git")

os.chdir(REPO)

KAGGLE_DATA = "/kaggle/input/datasets/asifaliansaree/ham10000-images/data"
WORK        = f"{REPO}/ham10000/data"
CONFIG_DIR  = f"{REPO}/ham10000/configs"
os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")
for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)
    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            import shutil
            shutil.rmtree(dst)
        else:
            os.remove(dst)
    os.symlink(src, dst)
print("Dataset linked.")

# ============================================================
# Git auth (token prompt — not stored in the notebook)
# ============================================================
GITHUB_TOKEN = getpass.getpass("GitHub token: ")
os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')
os.system(f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git')
os.system("git pull origin main")

# ============================================================
# Write the resnet18_v2_tuned config
# ============================================================
exp_name = "resnet18_v2_tuned"
ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir  = f"experiments/{exp_name}"
os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

cfg = {
    "seed": 42,
    "data": {
        "data_dir": WORK,
        "num_workers": 2,
        "img_size": 224,
        "augment": True,
    },
    "model": {
        "architecture": "resnet18",
        "num_classes": 7,
        "metadata_dim": 0,
        "pretrained": True,
        "dropout": 0.4,
        "freeze_epochs": 4,
    },
    "train": {
        "epochs": 45,
        "batch_size": 32,
        "lr": 7.94e-05,
        "weight_decay": 5e-4,
        "scheduler": "cosine",
        "gradient_accumulation": 1,
        "max_grad_norm": 1.0,
        "use_ema": True,
        "ema_decay": 0.999,
        "early_stopping_patience": 10,
        "use_weighted_sampler": False,
    },
    "loss": {
        "name": "weighted_focal",
        "focal_gamma": 2.0,
        "alpha_mode": "effective_num",
        "effective_num_beta": 0.999,
        "class_counts": [890, 5367, 412, 257, 891, 86, 114],
    },
    "output": {
        "checkpoint_dir": ckpt_dir,
    },
    "logging": {
        "experiment_name": exp_name,
        "save_dir": exp_dir,
        "checkpoint_dir": ckpt_dir,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"
with open(config_path, "w") as f:
    yaml.dump(cfg, f)
print(f"Created: {config_path}")

# ============================================================
# Train
# ============================================================
print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)
get_ipython().system(f'python {REPO}/ham10000/src/train.py --config {config_path}')

# ============================================================
# Commit + push results immediately (don't lose this run)
# ============================================================
print(f"\nCommitting results for {exp_name}...")
os.system(f"git add ham10000/experiments/{exp_name} ham10000/checkpoints/{exp_name} ham10000/configs/{exp_name}.yaml")
os.system(f'git commit -m "Add {exp_name} training results"')
result = os.system("git push origin main")
if result == 0:
    print(f"Pushed {exp_name} results successfully.")
else:
    print("Push failed — check token / auth.")

Cloning into 'Trustworthy-AI-for-Dermatology-Imaging'...


Linking dataset...
Dataset linked.


GitHub token:  ········


Already up to date.
Created: /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/resnet18_v2_tuned.yaml
TRAINING: resnet18_v2_tuned


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Device      : cuda
Experiment  : resnet18_v2_tuned
Architecture: resnet18
Metadata    : False
EMA         : True
Freeze ep   : 4
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 167MB/s]
Backbone frozen for first 4 epoch(s).
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:317: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 45 epochs — loss=weighted_focal, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:227: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.au

To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   6d6d500..37bfffb  main -> main


In [ ]:
import os
import subprocess
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ==========================================================
# Configuration
# ==========================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
EXP_NAME = "resnet18_v2_tuned"

# ---- GitHub Information ----
GITHUB_USERNAME = "AsifAliAnsaree"
GITHUB_EMAIL = "asifaliansari112@gmail.com"

# Leave empty ("") if you don't want to push automatically
GITHUB_TOKEN = "ghp_"

# ==========================================================
# Paths
# ==========================================================

LOG_PATH = os.path.join(
    REPO,
    "ham10000",
    "experiments",
    EXP_NAME,
    "training_log.csv",
)

OUT_PATH = os.path.join(
    REPO,
    "ham10000",
    "results",
    "figures",
    f"{EXP_NAME}_curves.png",
)

# ==========================================================
# Read Training Log
# ==========================================================

if not os.path.exists(LOG_PATH):
    raise FileNotFoundError(f"Training log not found:\n{LOG_PATH}")

df = pd.read_csv(LOG_PATH)

best_row = df.loc[df["val_bal_acc"].idxmax()]
best_epoch = int(best_row["epoch"])
best_acc = float(best_row["val_bal_acc"])

print("=" * 60)
print(f"Best Validation Balanced Accuracy : {best_acc:.4f}")
print(f"Best Epoch                        : {best_epoch}")
print("=" * 60)

# ==========================================================
# Plot Curves
# ==========================================================

fig, ax1 = plt.subplots(figsize=(10, 5.5))
ax2 = ax1.twinx()

# Loss curves
ax1.plot(
    df["epoch"],
    df["train_loss"],
    "--",
    linewidth=2,
    color="#E74C3C",
    label="Train Loss",
)

ax1.plot(
    df["epoch"],
    df["val_loss"],
    linewidth=2,
    color="#F39C12",
    label="Validation Loss",
)

# Validation Balanced Accuracy
ax2.plot(
    df["epoch"],
    df["val_bal_acc"],
    linewidth=2.5,
    color="#2874A6",
    label="Validation Balanced Accuracy",
)

# Best epoch
ax2.axvline(
    best_epoch,
    linestyle=":",
    color="gray",
    linewidth=1.5,
)

# Original baseline
BASELINE = 0.7964

ax2.axhline(
    BASELINE,
    linestyle="-.",
    linewidth=1.3,
    color="green",
    label=f"Baseline ({BASELINE:.4f})",
)

# Freeze -> Unfreeze
ax1.axvline(
    4.5,
    color="gray",
    linewidth=1,
)

ax1.text(
    4.6,
    ax1.get_ylim()[1] * 0.95,
    "Unfreeze",
    fontsize=8,
)

# Labels
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Weighted Focal Loss")
ax2.set_ylabel("Validation Balanced Accuracy")

ax1.set_title(
    f"{EXP_NAME}\nBest Validation Balanced Accuracy = {best_acc:.4f} (Epoch {best_epoch})"
)

ax1.grid(alpha=0.3)

# Legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center right",
)

plt.tight_layout()

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

plt.savefig(
    OUT_PATH,
    dpi=300,
    bbox_inches="tight",
)

plt.close()

print(f"\nFigure saved to:\n{OUT_PATH}")

# ==========================================================
# Git Section
# ==========================================================

os.chdir(REPO)

print("\nConfiguring Git...")

subprocess.run(["git", "config", "--global", "user.name", GITHUB_USERNAME])
subprocess.run(["git", "config", "--global", "user.email", GITHUB_EMAIL])

# Configure remote with token (optional)
if GITHUB_TOKEN.strip():

    remote_url = (
        f"https://{GITHUB_USERNAME}:{GITHUB_TOKEN}"
        f"@github.com/{GITHUB_USERNAME}/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

    subprocess.run(
        ["git", "remote", "set-url", "origin", remote_url],
        check=False,
    )

# ==========================================================
# Git Add
# ==========================================================

print("\nAdding files...")

subprocess.run(["git", "add", OUT_PATH])

# ==========================================================
# Git Commit
# ==========================================================

status = subprocess.run(
    ["git", "status", "--porcelain"],
    capture_output=True,
    text=True,
)

if status.stdout.strip() == "":
    print("\nNothing to commit.")
else:
    commit = subprocess.run(
        [
            "git",
            "commit",
            "-m",
            f"Add training curve for {EXP_NAME}",
        ]
    )

    if commit.returncode == 0:
        print("\nCommit successful.")
    else:
        print("\nCommit failed.")

# ==========================================================
# Git Push
# ==========================================================

if GITHUB_TOKEN.strip():

    print("\nPushing to GitHub...")

    push = subprocess.run(["git", "push", "origin", "main"])

    if push.returncode == 0:
        print("\nSuccessfully pushed to GitHub.")
    else:
        print(
            "\nPush failed.\n"
            "Check your GitHub username, token, or repository permissions."
        )

else:

    print(
        "\nNo GitHub token provided."
        "\nFigure has been saved locally."
        "\nSkipping push."
    )

print("\nDone!")

Best Validation Balanced Accuracy : 0.7416
Best Epoch                        : 40

Figure saved to:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/results/figures/resnet18_v2_tuned_curves.png

Configuring Git...

Adding files...

Nothing to commit.

Pushing to GitHub...

Successfully pushed to GitHub.

Done!


Everything up-to-date


In [3]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: repo + dataset linking (safe to re-run)
# ============================================================
REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

KAGGLE_DATA = "/kaggle/input/datasets/asifaliansaree/ham10000-images/data"
WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================
GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================
exp_name = "resnet18_v2_tuned_stable"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
# ============================================================
cfg = {
    "seed": 42,

    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },

    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },

    "loss": {
        "name": "weighted_focal",
        "alpha_mode": "effective_num",
        "effective_num_beta": 0.999,
        "focal_gamma": 1.5,
        "class_counts": [
            890,
            5367,
            412,
            257,
            891,
            86,
            114,
        ],
    },

    "model": {
        "architecture": "resnet18",
        "dropout": 0.35,
        "freeze_epochs": 4,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },

    "output": {
        "checkpoint_dir": ckpt_dir,
    },

    "train": {
        "batch_size": 32,
        "early_stopping_patience": 12,
        "ema_decay": 0.999,
        "epochs": 45,
        "gradient_accumulation": 1,
        "lr": 1e-4,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": False,
        "use_weighted_sampler": False,
        "weight_decay": 2e-4,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nCreated config:\n{config_path}")

# ============================================================
# Train
# ============================================================
print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)

get_ipython().system(
    f'python {REPO}/ham10000/src/train.py --config {config_path}'
)

# ============================================================
# Save Results to GitHub
# ============================================================
print("\nSaving results to GitHub...")

os.system(
    f"git add ham10000/checkpoints/{exp_name} "
    f"ham10000/experiments/{exp_name} "
    f"ham10000/configs/{exp_name}.yaml"
)

os.system(f'git commit -m "Add {exp_name} training results"')

result = os.system("git push origin main")

if result == 0:
    print("Results pushed successfully!")
else:
    print("Git push failed. Check your token or GitHub permissions.")

Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/resnet18_v2_tuned_stable.yaml
TRAINING: resnet18_v2_tuned_stable


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Device      : cuda
Experiment  : resnet18_v2_tuned_stable
Architecture: resnet18
Metadata    : False
EMA         : False
Freeze ep   : 4
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 146MB/s]
Backbone frozen for first 4 epoch(s).
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:317: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 45 epochs — loss=weighted_focal, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:227: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torc

To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   d9c7831..2f70bc2  main -> main


In [4]:
import os
import getpass
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --------------------------------------------------
# Git Authentication
# --------------------------------------------------
GITHUB_TOKEN = getpass.getpass("GitHub Token: ")

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
os.chdir(REPO)

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

os.system("git pull origin main")

# --------------------------------------------------
# Experiment
# --------------------------------------------------
exp_name = "resnet18_v2_tuned_stable"

log_path = f"ham10000/experiments/{exp_name}/training_log.csv"

assert os.path.exists(log_path), f"{log_path} not found."

log = pd.read_csv(log_path)

# --------------------------------------------------
# Create Figure
# --------------------------------------------------
os.makedirs("ham10000/results/figures", exist_ok=True)

fig, ax1 = plt.subplots(figsize=(10,5))
ax2 = ax1.twinx()

ax1.plot(
    log["epoch"],
    log["train_loss"],
    linewidth=2,
    linestyle="--",
    label="Train Loss"
)

ax1.plot(
    log["epoch"],
    log["val_loss"],
    linewidth=2,
    label="Validation Loss"
)

ax2.plot(
    log["epoch"],
    log["val_bal_acc"],
    linewidth=2.5,
    label="Balanced Accuracy"
)

best_epoch = int(log.loc[log["val_bal_acc"].idxmax(), "epoch"])
best_acc = float(log["val_bal_acc"].max())

ax2.scatter(best_epoch, best_acc, s=80)

ax2.annotate(
    f"Best={best_acc:.4f}",
    (best_epoch, best_acc),
    xytext=(best_epoch + 1, best_acc - 0.03),
    arrowprops=dict(arrowstyle="->")
)

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax2.set_ylabel("Balanced Accuracy")

plt.title(f"{exp_name}\nTraining Curves")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")

plt.tight_layout()

png_path = f"ham10000/results/figures/{exp_name}_training_curves.png"

plt.savefig(png_path, dpi=300)
plt.close()

print("Saved:", png_path)

# --------------------------------------------------
# Push to GitHub
# --------------------------------------------------
os.system(f"git add {png_path}")

os.system(
    f'git commit -m "Add training curves for {exp_name}"'
)

result = os.system("git push origin main")

if result == 0:
    print("\nPNG pushed successfully!")
else:
    print("\nPush failed.")

GitHub Token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Saved: ham10000/results/figures/resnet18_v2_tuned_stable_training_curves.png
[main 0530594] Add training curves for resnet18_v2_tuned_stable
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 ham10000/results/figures/resnet18_v2_tuned_stable_training_curves.png

PNG pushed successfully!


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   2f70bc2..0530594  main -> main
